<a href="https://colab.research.google.com/github/koderlad/M507D---Methods-of-Prediction/blob/main/M507D_Week_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Importing Dependencies**

In [1]:
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier

## **Loading Dataset**

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/m-mahdavi/teaching/refs/heads/main/datasets/mnist.csv')

### **Splitting Dataset**

In [3]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(['class', 'id'], axis=1), df['class'], test_size=0.2, stratify=df['class'])

## **Quick Exploration**

In [4]:
print(f"Train Data Shape: {X_train.shape} \nTest Data Shape: {X_test.shape}")

Train Data Shape: (3200, 784) 
Test Data Shape: (800, 784)


In [5]:
X_train.dropna(inplace=True)
X_train.drop_duplicates(inplace=True)
X_test.dropna(inplace=True)
X_test.drop_duplicates(inplace=True)

In [6]:
print(f"Train Data Shape: {X_train.shape} \nTest Data Shape: {X_test.shape}")

Train Data Shape: (3200, 784) 
Test Data Shape: (800, 784)


Training and Test dataset does not have null or duplicate values.

### **Class Balance**

In [7]:
y_train.value_counts()

,count
class,
1,389
7,341
8,333
3,333
6,313
2,312
0,301
4,295
9,293


Kind of balanced, "Accuracy" can be used as evaluation metric.

## **Baseline Model**

I am using the same classifier but without Hyper-Parameter finetuning, just the default values it already comes assigned with. And no Scaling of the values.

In [8]:
baseline = MLPClassifier()

### **Cross Validating**
Making sure that the model's accuracy isn't a one time thing and isn't a good memorizer but a good classfier.

In [9]:
cv_score_baseline = cross_val_score(baseline, X_train, y_train, cv=5, scoring='accuracy')

In [10]:
print(f"Cross Val Score (Baseline): {cv_score_baseline}")
print(f"Average Accuracy: {cv_score_baseline.mean()}\n Standard Deviation: {cv_score_baseline.std()}")

Cross Val Score (Baseline): [0.8578125 0.85625   0.8609375 0.8640625 0.8515625]
Average Accuracy: 0.8581249999999999
 Standard Deviation: 0.004238956239453287


### **Actual Training**

In [11]:
baseline.fit(X_train, y_train)

MLPClassifier()

In [12]:
y_pred_baseline = baseline.predict(X_test)

In [13]:
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
print(f"Baseline Model Accuracy on Test Data is {round(baseline_accuracy, 4)}")

Baseline Model Accuracy on Test Data is 0.875


## **Now with Hyper-parameter fine-tuning**

In [14]:
#Hard Refreshing Classifier
mlp = MLPClassifier()

In [15]:
pipe = Pipeline(steps=[
    ('standardscaler', StandardScaler()),
    ('model', mlp)
])

### **With Scaling Features**

### **Cross Validating**

In [16]:
cv_score_scaled = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')

In [17]:
print(f"Cross Val Score (Baseline): {cv_score_scaled}")
print(f"Average Accuracy: {cv_score_scaled.mean()}\n Standard Deviation: {cv_score_scaled.std()}")

Cross Val Score (Baseline): [0.915625  0.9109375 0.928125  0.9234375 0.91875  ]
Average Accuracy: 0.9193749999999999
 Standard Deviation: 0.005978477021784067


### **Actual Training**

In [18]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('model', MLPClassifier())])

In [19]:
y_pred_scaled = pipe.predict(X_test)

In [20]:
scaled_accuracy = accuracy_score(y_test, y_pred_scaled)
print(f"Accuracy (Scaled Features) on Test Data is {round(scaled_accuracy, 4)}")

Accuracy (Scaled Features) on Test Data is 0.91


## **Searching the best Hyperparameters**

In [21]:
#Hard Refreshing Classifier and pipeline
mlp_r = MLPClassifier()
pipe_r = Pipeline(steps=[
    ('standardscaler', StandardScaler()),
    ('model', mlp_r)
])

In [22]:
params = {"MLP": {
    'model__hidden_layer_sizes': [(100,), (50, 50)],
    'model__activation': ['relu', 'tanh'],
    'model__solver': ['lbfgs', 'adam'],
    'model__learning_rate_init': [0.001, 0.01],
    # 'model__max_iter': [200, 400, 700],
    'model__tol': [0.0001, 0.001, 0.01],
    'model__early_stopping': [True],
    'model__validation_fraction': [0.1, 0.15]
}}

In [23]:
search = GridSearchCV(
    estimator = pipe_r,
    param_grid = params['MLP'],
    scoring = 'accuracy',
    cv = 5,
    n_jobs = -1,
)

In [24]:
search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('standardscaler', StandardScaler()),
                                       ('model', MLPClassifier())]),
             n_jobs=-1,
             param_grid={'model__activation': ['relu', 'tanh'],
                         'model__early_stopping': [True],
                         'model__hidden_layer_sizes': [(100,), (50, 50)],
                         'model__learning_rate_init': [0.001, 0.01],
                         'model__solver': ['lbfgs', 'adam'],
                         'model__tol': [0.0001, 0.001, 0.01],
                         'model__validation_fraction': [0.1, 0.15]},
             scoring='accuracy')

In [25]:
print("Best Score (Accuracy): ", round(search.best_score_, 4))
print("Best Params: ", search.best_params_)

Best Score (Accuracy):  0.9178
Best Params:  {'model__activation': 'relu', 'model__early_stopping': True, 'model__hidden_layer_sizes': (100,), 'model__learning_rate_init': 0.01, 'model__solver': 'adam', 'model__tol': 0.001, 'model__validation_fraction': 0.1}


In [26]:
beast_model = search.best_estimator_

In [27]:
y_pred = beast_model.predict(X_test)

In [28]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy on Test Data is {round(accuracy,4)}")

Accuracy on Test Data is 0.9087


## **Comparison**

In [29]:
cr_baseline = classification_report(y_test, y_pred_baseline)
cr_scaled = classification_report(y_test, y_pred_scaled)
cr = classification_report(y_test, y_pred)

In [30]:
print(cr_baseline, cr_scaled, cr)

              precision    recall  f1-score   support

           0       0.89      0.97      0.93        75
           1       0.92      0.97      0.94        97
           2       0.98      0.74      0.85        78
           3       0.85      0.88      0.87        84
           4       0.93      0.76      0.84        74
           5       0.92      0.79      0.85        73
           6       0.85      0.94      0.89        78
           7       0.88      0.93      0.90        85
           8       0.84      0.83      0.84        83
           9       0.74      0.90      0.81        73

    accuracy                           0.88       800
   macro avg       0.88      0.87      0.87       800
weighted avg       0.88      0.88      0.87       800
               precision    recall  f1-score   support

           0       0.99      0.99      0.99        75
           1       0.92      0.98      0.95        97
           2       0.91      0.78      0.84        78
           3       0.89 